# 3-approach retrieval comparison — `faiss_index_cases` (bge-base)

Compares three retrieval stages on the **cases** retriever, using the corpus-grounded
gold set `golden_queries_bge.jsonl` (480 queries, 100% coverage). No API / LLM calls.

| Stage | What it does |
|---|---|
| **dense** | `store.similarity_search_with_score` — bge embeddings, FAISS L2 only |
| **hybrid** | the app's `hybrid_retrieval.hybrid_search` — BM25 + dense fused by weighted RRF (0.6 / 0.4, k=60) |
| **hybrid+cit** | hybrid + **exact case-citation anchor**: if the query names a citation, promote that case's best chunk to rank 1 |

Cases are ranked by each method's own hit order (first occurrence per citation), so
RRF fusion and citation anchoring are reflected faithfully. Metrics: Recall@1/5/10,
MRR@10, nDCG@10 with bootstrap 95% CIs, per-query-type breakdown, and gate AUC.

**Note:** the citation anchor fires only on queries containing a recognizable citation
(the 150 `citation`-type queries); `case_name` and `issue` queries are unchanged from
hybrid. This is a case-citation match, distinct from the app's statute section-number
`_citation_boost` (which applies to the law index).

Writes `eval_results_dense.json`, `eval_results_hybrid.json`, `eval_results_citation.json`,
`eval_summary_3way.txt`.


In [1]:
import json, re, os, sys, time
import numpy as np
import faiss, torch
sys.path.insert(0, os.path.abspath(".."))
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import hybrid_retrieval as hr

BGE_DIR = "../faiss_index_cases"; GOLD = "golden_queries_bge.jsonl"
MODEL = "BAAI/bge-base-en-v1.5"; KIND = "cases"
FETCH_K = 100; TOP_K = 10; SEED = 42; N_BOOT = 2000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
lines = []
def say(s=""):
    print(s, flush=True); lines.append(s)


In [2]:
def canon(s):
    s = re.sub(r"(?i)citation\s*name\s*:", " ", s or "")
    s = re.split(r"(?i)bookmark", s)[0]
    return re.sub(r"[^a-z0-9]", "", s.lower())

def group_cases(pairs, top_k):
    """Rank distinct cases by FIRST occurrence in the method's ordered hit list."""
    seen = set(); ordered = []
    for doc, dist in pairs:
        k = canon((doc.metadata or {}).get("citation", ""))
        if not k or k in seen: continue
        seen.add(k); ordered.append((k, 1.0 - float(dist) / 2.0))
        if len(ordered) >= top_k: break
    return ordered

def make_records(rows, ranked_all, index_keys):
    res = []
    for row, ranked in zip(rows, ranked_all):
        keys = [k for k, _ in ranked]
        rec = {"id": row["id"], "query": row["query"], "query_type": row["query_type"],
               "answerable": row["answerable"],
               "top_score": float(ranked[0][1]) if ranked else -1.0,
               "top_cases": [{"citation_norm": k, "cos": round(s, 4)} for k, s in ranked[:5]]}
        if row["answerable"]:
            gold = {canon(c) for c in row["relevant_citations"]} - {""}
            rec["gold"] = sorted(gold); rec["gold_in_index"] = bool(gold & index_keys)
            rec["rank"] = next((i + 1 for i, k in enumerate(keys) if k in gold), None)
        res.append(rec)
    return res

def retrieval_metrics(recs):
    ranks = [r["rank"] for r in recs]
    def hit(K): return [1.0 if (r is not None and r <= K) else 0.0 for r in ranks]
    return {"Recall@1": hit(1), "Recall@5": hit(5), "Recall@10": hit(10),
            "MRR@10": [1.0 / r if (r is not None and r <= TOP_K) else 0.0 for r in ranks],
            "nDCG@10": [1.0 / np.log2(r + 1) if (r is not None and r <= TOP_K) else 0.0 for r in ranks]}

def bootstrap_ci(vals, rng):
    v = np.asarray(vals, float)
    if len(v) == 0: return 0.0, 0.0, 0.0
    idx = rng.integers(0, len(v), size=(N_BOOT, len(v))); means = v[idx].mean(axis=1)
    return v.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

def show_block(title, recs, rng):
    say(f"\n{title}  (n={len(recs)})")
    if not recs: say("  no queries"); return {}
    m = {}
    for name, vals in retrieval_metrics(recs).items():
        mu, lo, hi = bootstrap_ci(vals, rng); m[name] = float(mu)
        say(f"  {name:10s} {mu:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")
    return m

def auc(pos, neg):
    pos, neg = np.asarray(pos), np.asarray(neg)
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    return float((pos[:, None] > neg[None, :]).mean() + 0.5 * (pos[:, None] == neg[None, :]).mean())


In [3]:
say("Loading bge store (faiss_index_cases) ...")
emb = HuggingFaceEmbeddings(model_name=MODEL, model_kwargs={"device": DEVICE},
                            encode_kwargs={"normalize_embeddings": True, "batch_size": 16})
store = FAISS.load_local(BGE_DIR, emb, allow_dangerous_deserialization=True)
index_keys = {canon((d.metadata or {}).get("citation", "")) for d in store.docstore._dict.values()} - {""}
hr._get_entry(KIND, store)   # loads BM25 from cache (ntotal matches)
docid_to_row = {doc_id: row for row, doc_id in store.index_to_docstore_id.items()}
cit_map = {}
for doc_id, doc in store.docstore._dict.items():
    k = canon((doc.metadata or {}).get("citation", ""))
    if k: cit_map.setdefault(k, []).append((doc_id, docid_to_row.get(doc_id)))
say(f"store ready: {store.index.ntotal} vectors, {len(index_keys)} cases, cit_map {len(cit_map)} citation keys")

CIT_RE = re.compile(r"\b(?:19|20)\d{2}\s+[A-Za-z][A-Za-z\.]*\s+\d+(?:\s+[A-Z][A-Z\-]+)?")

def retrieve_dense(q):
    return store.similarity_search_with_score(q, k=FETCH_K)

def retrieve_hybrid(q):
    pairs = hr.hybrid_search(KIND, store, q, FETCH_K)
    return pairs if pairs is not None else retrieve_dense(q)

def anchor_citation(q, hits):
    entries = cit_map.get(canon(q))
    if not entries:
        m = CIT_RE.search(q)
        if m: entries = cit_map.get(canon(m.group(0)))
    if not entries: return hits
    qvec = np.asarray(emb.embed_query(q), dtype=np.float32)
    scored = []
    for doc_id, row in entries:
        if row is None: continue
        scored.append((store.docstore._dict[doc_id], hr._l2_distance(store, qvec, row)))
    if not scored: return hits
    scored.sort(key=lambda x: x[1])
    return scored[:1] + hits   # promote cited case's best chunk to rank 1


Loading bge store (faiss_index_cases) ...
BM25 cases index loaded from cache (209,168 chunks).
store ready: 209168 vectors, 15627 cases, cit_map 15627 citation keys


In [4]:
rows = [json.loads(l) for l in open(GOLD, encoding="utf-8") if l.strip()]
queries = [r["query"] for r in rows]
ranked = {"dense": [], "hybrid": [], "citation": []}
t0 = time.time()
for i, q in enumerate(queries):
    h = retrieve_hybrid(q)
    ranked["dense"].append(group_cases(retrieve_dense(q), TOP_K))
    ranked["hybrid"].append(group_cases(h, TOP_K))
    ranked["citation"].append(group_cases(anchor_citation(q, h), TOP_K))
    if (i + 1) % 100 == 0: say(f"  {i+1}/{len(queries)} queries  ({time.time()-t0:.0f}s)")
say(f"search done in {time.time()-t0:.0f}s")

RES = {}
for app in ("dense", "hybrid", "citation"):
    recs = make_records(rows, ranked[app], index_keys)
    json.dump(recs, open(f"eval_results_{app}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    RES[app] = recs

rng = np.random.default_rng(SEED)
SUM = {}
for app in ("dense", "hybrid", "citation"):
    recs = RES[app]
    ans = [r for r in recs if r["answerable"]]; un = [r for r in recs if not r["answerable"]]
    ceil = [r for r in ans if r["gold_in_index"]]
    say(f"\n########## {app.upper()}  (ceiling n={len(ceil)}, coverage 100%) ##########")
    overall = show_block("All answerable", ceil, rng)
    types = {}
    for qt in sorted({r["query_type"] for r in ceil}):
        types[qt] = show_block(qt, [r for r in ceil if r["query_type"] == qt], rng)
    A = auc([r["top_score"] for r in ans], [r["top_score"] for r in un])
    say(f"\ngate AUC {A:.3f}")
    SUM[app] = {"overall": overall, "types": types, "auc": A, "n": len(ceil)}


  100/480 queries  (98s)
  200/480 queries  (172s)
  300/480 queries  (260s)
  400/480 queries  (337s)
search done in 416s

########## DENSE  (ceiling n=450, coverage 100%) ##########

All answerable  (n=450)
  Recall@1   0.164   95% CI [0.131, 0.198]
  Recall@5   0.213   95% CI [0.176, 0.251]
  Recall@10  0.238   95% CI [0.198, 0.280]
  MRR@10     0.185   95% CI [0.152, 0.218]
  nDCG@10    0.197   95% CI [0.163, 0.232]

case_name  (n=150)
  Recall@1   0.067   95% CI [0.033, 0.107]
  Recall@5   0.100   95% CI [0.060, 0.153]
  Recall@10  0.113   95% CI [0.067, 0.167]
  MRR@10     0.077   95% CI [0.039, 0.121]
  nDCG@10    0.086   95% CI [0.045, 0.128]

citation  (n=150)
  Recall@1   0.013   95% CI [0.000, 0.033]
  Recall@5   0.020   95% CI [0.000, 0.047]
  Recall@10  0.040   95% CI [0.013, 0.073]
  MRR@10     0.018   95% CI [0.002, 0.039]
  nDCG@10    0.023   95% CI [0.007, 0.046]

issue  (n=150)
  Recall@1   0.413   95% CI [0.340, 0.487]
  Recall@5   0.520   95% CI [0.440, 0.600]
  Rec

In [5]:
say("\n########## 3-WAY COMPARISON  (bge faiss_index_cases, grounded gold) ##########")
say(f"{'metric':22}{'dense':>10}{'hybrid':>10}{'hybrid+cit':>12}")
for key in ("Recall@1", "Recall@5", "Recall@10", "MRR@10", "nDCG@10"):
    say(f"{key:22}{SUM['dense']['overall'][key]:>10.3f}{SUM['hybrid']['overall'][key]:>10.3f}{SUM['citation']['overall'][key]:>12.3f}")
say(f"{'gate AUC':22}{SUM['dense']['auc']:>10.3f}{SUM['hybrid']['auc']:>10.3f}{SUM['citation']['auc']:>12.3f}")
say("\nRecall@10 by query type:")
say(f"{'type':14}{'dense':>10}{'hybrid':>10}{'hybrid+cit':>12}")
for qt in ("citation", "case_name", "issue"):
    say(f"{qt:14}{SUM['dense']['types'][qt]['Recall@10']:>10.3f}{SUM['hybrid']['types'][qt]['Recall@10']:>10.3f}{SUM['citation']['types'][qt]['Recall@10']:>12.3f}")
say("\nRecall@1 by query type:")
say(f"{'type':14}{'dense':>10}{'hybrid':>10}{'hybrid+cit':>12}")
for qt in ("citation", "case_name", "issue"):
    say(f"{qt:14}{SUM['dense']['types'][qt]['Recall@1']:>10.3f}{SUM['hybrid']['types'][qt]['Recall@1']:>10.3f}{SUM['citation']['types'][qt]['Recall@1']:>12.3f}")
say("\nStages: dense=FAISS only | hybrid=app BM25+dense RRF | hybrid+cit=hybrid+exact citation anchor")
say("Citation anchor fires only on the 150 citation-type queries; others unchanged from hybrid.")
open("eval_summary_3way.txt", "w", encoding="utf-8").write("\n".join(lines) + "\n")
say("\nSaved: eval_results_dense.json, eval_results_hybrid.json, eval_results_citation.json, eval_summary_3way.txt")



########## 3-WAY COMPARISON  (bge faiss_index_cases, grounded gold) ##########
metric                     dense    hybrid  hybrid+cit
Recall@1                   0.164     0.182       0.509
Recall@5                   0.213     0.238       0.562
Recall@10                  0.238     0.262       0.582
MRR@10                     0.185     0.210       0.535
nDCG@10                    0.197     0.222       0.546
gate AUC                   0.846     0.852       0.678

Recall@10 by query type:
type               dense    hybrid  hybrid+cit
citation           0.040     0.040       1.000
case_name          0.113     0.127       0.127
issue              0.560     0.620       0.620

Recall@1 by query type:
type               dense    hybrid  hybrid+cit
citation           0.013     0.020       1.000
case_name          0.067     0.087       0.087
issue              0.413     0.440       0.440

Stages: dense=FAISS only | hybrid=app BM25+dense RRF | hybrid+cit=hybrid+exact citation anchor
Citation anc